# imports

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
from itertools import combinations
import numpy as np

# Mapping tasks to animations

In [8]:

task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

In [9]:
def decode_task(task_id) -> dict:
    tid = str(task_id)
    code = task_mapping.get(tid, "")
    if not code:
        return {"type": "Unknown", "delay": None, "ease": None, "direction": None, "code": None}

    family = code[0]
    fam_info = animation_mapping.get(family, {})
    result = {"type": fam_info.get("type", "Unknown"), "delay": None, "ease": None, "direction": None, "code": code}

    if "null" in code.lower():
        return result


    digits = [d for d in code[1:] if d.isdigit()]
    while len(digits) < 3:
        digits.append("0")
    idx_delay, idx_ease, idx_dir = (int(digits[0]), int(digits[1]), int(digits[2]))

    def pick(options, i):
        """options can be list or dict; return None/UNSPEC if not available."""
        if options is None:
            return None
        if isinstance(options, dict):
            return options.get(i, f"UNSPEC({i})")
        if isinstance(options, list):
            return options[i] if 0 <= i < len(options) else f"UNSPEC({i})"
        return None

    result["delay"]     = pick(fam_info.get("delay"),     idx_delay)
    result["ease"]      = pick(fam_info.get("ease"),      idx_ease)
    result["direction"] = pick(fam_info.get("direction"), idx_dir)

    return result

In [4]:
decode_task("2.6")

{'type': 'BOUNCE',
 'delay': '0.3s',
 'ease': 'Ease-out',
 'direction': 'Pulsing',
 'code': 'B013'}

# Load and clean csv

In [14]:
fix_path = "outputs/fixations_within_tasks.csv"
gaze_path = "outputs/gaze_within_tasks.csv"

fx = pd.read_csv(fix_path)
gz = pd.read_csv(gaze_path)


/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_37441/2067053192.py:5: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  gz = pd.read_csv(gaze_path)


dropped invalid data and values outside range of ]0, 1[ for fixations.

duration have to be longer than 0.


In [15]:
fx = fx.copy()

num_cols_fx = ["FPOGX","FPOGY","FPOGS","FPOGD","FPOGV","BKID","BKDUR","LPMM","RPMM", "SACCADE_MAG","SACCADE_DIR"]
for c in num_cols_fx:
    if c in fx.columns:
        fx[c] = pd.to_numeric(fx[c], errors="coerce")

fx["Task Id"] = fx["Task Id"].astype(str).str.strip()

if "FPOGV" in fx.columns:
    fx = fx[fx["FPOGV"] == 1].copy()

fx = fx.dropna(subset=["FPOGX","FPOGY","FPOGD"]).copy()
fx = fx[(fx["FPOGX"].between(0,1)) & (fx["FPOGY"].between(0,1)) & (fx["FPOGD"] > 0)].copy()


drop non-saccades, normalize directions

In [17]:
gz = gz.copy()

num_cols_gz = ["FPOGX","FPOGY","TIME","SACCADE_MAG","SACCADE_DIR","BKID","BKDUR","LPMM","RPMM"]
for c in num_cols_gz:
    if c in gz.columns:
        gz[c] = pd.to_numeric(gz[c], errors="coerce")

gz["Task Id"] = gz["Task Id"].astype(str).str.strip()

gz = gz.dropna(subset=["Task Id"]).copy()

if "FPOGX" in gz.columns and "FPOGY" in gz.columns:
    gz = gz.dropna(subset=["FPOGX","FPOGY"]).copy()
    gz = gz[(gz["FPOGX"].between(0,1)) & (gz["FPOGY"].between(0,1))].copy()

if "SACCADE_MAG" in gz.columns:
    gz = gz[(gz["SACCADE_MAG"].notna()) & (gz["SACCADE_MAG"] > 0)].copy()

if "SACCADE_DIR" in gz.columns:
    gz["SACCADE_DIR"] = (gz["SACCADE_DIR"] % 360).where(gz["SACCADE_DIR"].notna())


attach tasks durations

In [23]:
times_path = "/Users/ferolahima/Desktop/gazepoint_project/Export-Video Mappings.xlsx - Foglio1.csv"
tasks_all = pd.read_csv(times_path, sep=None, engine="python")

dur_lookup = tasks_all[["Video Name","User ID","Task Id","Duration in seconds"]].copy()
dur_lookup["Task Id"] = dur_lookup["Task Id"].astype(str).str.strip()
dur_lookup.rename(columns={"Duration in seconds":"task_duration"}, inplace=True)

fx_merged = fx.merge(
    dur_lookup,
    left_on=["Video Name (folder)","User ID (file)","Task Id"],
    right_on=["Video Name","User ID","Task Id"],
    how="left"
)
gz_merged = gz.merge(
    dur_lookup,
    left_on=["Video Name (folder)","User ID (file)","Task Id"],
    right_on=["Video Name","User ID","Task Id"],
    how="left"
)

fx_merged.drop(columns=["Video Name","User ID"], inplace=True, errors="ignore")
gz_merged.drop(columns=["Video Name","User ID"], inplace=True, errors="ignore")

fx_merged

,Video Name (folder),User ID (file),Source,MEDIA_NAME,TIME,TIMETICK(f=10000000),FPOGX,FPOGY,FPOGS,FPOGD,...,LPMM,LPMMV,RPMM,RPMMV,AOI,SACCADE_MAG,SACCADE_DIR,Unnamed: 21,Task Id,task_duration
0,P10_S1_ITA,User 0,fixation,P10S1,193.19971,92688076792,0.10351,0.63810,192.98633,0.21338,...,2.62316,1,2.80057,1,NaN,1842.90088,90.51694,NaN,2.1,6
1,P10_S1_ITA,User 0,fixation,P10S1,193.31494,92689227578,0.12530,0.27764,193.26562,0.04932,...,2.82945,1,2.89738,1,NaN,783.07672,83.86610,NaN,2.1,6
2,P10_S1_ITA,User 0,fixation,P10S1,193.79102,92693991750,0.10916,0.30957,193.59424,0.19678,...,3.11769,1,2.85108,1,NaN,1738.81519,246.93942,NaN,2.1,6
3,P10_S1_ITA,User 0,fixation,P10S1,194.05420,92696620968,0.09338,0.46637,193.85693,0.19727,...,3.44904,1,3.20035,1,NaN,344.06589,259.85645,NaN,2.1,6
4,P10_S1_ITA,User 0,fixation,P10S1,194.36621,92699742043,0.14280,0.07088,194.11963,0.24658,...,3.60240,1,3.27536,1,NaN,875.08350,77.47518,NaN,2.1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14007,P9_S2_ITA,User 0,fixation,S1IT,1689.44751,46974149842,0.25622,0.49748,1688.59302,0.85449,...,4.39059,0,4.28457,0,NaN,582.68079,169.62737,NaN,2.4,6
14008,P9_S2_ITA,User 0,fixation,S1IT,1689.95654,46979240802,0.31371,0.50256,1689.66089,0.29565,...,4.02804,1,4.39130,1,NaN,221.03407,357.15448,NaN,2.4,6
14009,P9_S2_ITA,User 0,fixation,S1IT,1690.33435,46983018864,0.38756,0.58757,1690.02234,0.31201,...,4.12645,1,4.18714,1,NaN,337.84143,327.07681,NaN,2.4,6
14010,P9_S2_ITA,User 0,fixation,S1IT,1690.63013,46985976430,0.25318,0.64701,1690.40027,0.22986,...,4.91400,1,4.73686,1,NaN,531.75183,193.97202,NaN,2.4,6


# Analysis

In [27]:
print(fx.columns.tolist())

['Video Name (folder)', 'User ID (file)', 'Source', 'MEDIA_NAME', 'TIME', 'TIMETICK(f=10000000)', 'FPOGX', 'FPOGY', 'FPOGS', 'FPOGD', 'FPOGID', 'FPOGV', 'BPOGX', 'BPOGY', 'BPOGV', 'BKID', 'BKDUR', 'LPMM', 'LPMMV', 'RPMM', 'RPMMV', 'AOI', 'SACCADE_MAG', 'SACCADE_DIR', 'Unnamed: 21', 'Task Id']


In [28]:
fx_stats = (
    fx_merged.groupby(["Video Name (folder)","User ID (file)","Task Id"], dropna=False)["FPOGD"]
      .agg(mean_dur="mean", median_dur="median", total_dur="sum", std_dur="std", n_fixations="count")
      .reset_index()
      .merge(dur_lookup, on=["Video Name (folder)","User ID (file)","Task Id"], how="left")
      .rename(columns={"task_duration":"task_duration"})  # already named
)

fx_stats

KeyError: 'Video Name (folder)'